In [ ]:
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv

In [ ]:
#Building necessary files for github--

def create_initial_files():
  #Building .evn.example file()
  env_example_content = "ACLED_EMAIL=your_email_here\nACLED_PASSWORD=your_password_here"
  if not os.path.exists(".env.example"):
    with open(".env.example", "w") as f:
      f.write(env_example_content)
    print(".env.example created.")

# Building .gitignore
    gitignore_content = ".env\nraw_data_output.csv\n__pycache__/\n.ipynb_checkpoints/\n*.pyc"
    if not os.path.exists(".gitignore"):
        with open(".gitignore", "w") as f:
            f.write(gitignore_content)
        print(" .gitignore created!")



# --- 2. Reading environmental variables ---
load_dotenv()


import os
from dotenv import load_dotenv

def get_credentials():
    """
    This is universal function to work in all ide.
    """
    email = None
    password = None

    # 1.checking if there is pswd and email exits in google colab
    try:
        from google.colab import userdata
        # if it is in the google colab, it will take email and pswd from secret icon direclty
        try:
            email = userdata.get('ACLED_EMAIL')
            password = userdata.get('ACLED_PASSWORD')
            if email and password:
                print("Using credentials from Google Colab Secrets.")
        except Exception:
            pass
    except ImportError:
        pass

    # 2. finding email and pswd in .env file
    if not email or not password:
        load_dotenv() # upload .env file
        email = os.environ.get('ACLED_EMAIL')
        password = os.environ.get('ACLED_PASSWORD')

        if email and password:
            print("Using credentials from Environment Variables/.env file.")

    # 3. Waring if they are not found anywhere
    if not email or not password:
        print("Warning: ACLED_EMAIL or ACLED_PASSWORD not found!")
        print("Fix: Set them in Colab Secrets OR create a .env file locally.")

    return email, password

MY_EMAIL, MY_PASSWORD = get_credentials()

if MY_EMAIL:
    print(f"Logged in as: {MY_EMAIL}")

In [ ]:
# --- 3. Function to extract data at ACLED API ---
def download_acled_api_data(country='Myanmar'):
   # A. Get Access Token
    auth_url = "https://acleddata.com/oauth/token"
    auth_payload = {
        'username': MY_EMAIL,
        'password': MY_PASSWORD,
        'grant_type': 'password',
        'client_id': 'acled'
    }

    try:
      auth_res = requests.post(auth_url, data = auth_payload)
      data = auth_res.json()
      token = data.get('access_token')
      print("Token have gotten now.")
    except:
      token = None

    if not token:
      print("Login failed. Check your credentails.")
      return

    return token

download_acled_api_data()



In [ ]:
token = download_acled_api_data()
country = "Myanmar"
# B. Fetch Data (Pagination System , 5000 rows each time)
all_data = []
page = 1
headers = {'Authorization': f'Bearer {token}'}

print(f" Starting download for {country}...")

while True:
    params = {
        'country': country,
        'limit': 5000,
        'page': page,
        'event_date': '2021-02-01',
        'event_date_where': '>='
    }

    response = requests.get(
        "https://acleddata.com/api/acled/read",
        params=params,
        headers=headers
    )

    if response.status_code == 200:
        batch = response.json().get('data', [])
        if not batch:
            break
        all_data.extend(batch)
        print(f"Page {page} done (Total: {len(all_data)} rows)")
        page += 1
        time.sleep(0.5)
    else:
        print(f"API Error: {response.status_code}")
        break

In [ ]:
#Storing as csv file
if all_data:
  df = pd.DataFrame(all_data)
  filename = 'raw_data_output.csv'
  df.to_csv(filename, index = False)
  print(f"Successfully saved to {filename}")
else:
  print("Not data found to save")

In [ ]:
if __name__ == "__main__":
  create_initial_files()



In [ ]:
#Checking the file name
if os.path.exists(filename):
  print(f"{filename} exists already.")
  print("We can proceed directly to EDA")

else:
  print(f"{filename} not foound. Starting fresh download.")
  download_acled_api_data




In [ ]:
df = pd.read_csv("raw_data_output.csv")

In [ ]:
df = df.convert_dtypes()
df.info()


In [ ]:
import pandas as pd
import numpy as np
import re

# ==========================================
# ၁။ DYNAMIC SETTINGS
# ==========================================
THRESHOLDS = {100000: "Massive", 1000: "Large", 100: "Medium", 0: "Small"}
MULTIPLIERS = {'million': 1000000, 'thousand': 1000, 'hundred': 100, 'dozen': 12, 'ten': 10}
IGNORE_KEYWORDS = ['no report', 'report', 'nan', 'n/a', 'unknown', 'no crowd data']

# ==========================================
# ၂။ DYNAMIC FUNCTION (to make a new column called crowd size category like small, medium, large from original crowd tag
# ==========================================
def truly_dynamic_categorizer(text):
    text = str(text).lower().strip()
    if any(x in text for x in IGNORE_KEYWORDS) or text == "":
        return "No Data"

    val = 1
    digit_match = re.search(r'\d+', text.replace(',', ''))
    if digit_match:
        val = int(digit_match.group())
    elif 'tens' in text: val = 10
    elif 'hundreds' in text and 'thousands' not in text: val = 100

    calculated_value = val
    for unit, factor in MULTIPLIERS.items():
        if unit in text:
            if unit == 'thousand' and 'hundred' in text: calculated_value = 100 * factor
            elif unit == 'thousand' and 'ten' in text: calculated_value = 10 * factor
            else: calculated_value = val * factor
            break

    for limit, label in sorted(THRESHOLDS.items(), reverse=True):
        if calculated_value >= limit: return label
    return "Small"

# ==========================================
# ၃။ FULL CLEANING PROCESS
# ==========================================

# (A) Reading data and basic cleaning
df = pd.read_csv("raw_data_output.csv").convert_dtypes().drop_duplicates()

# (B) DATE CONVERSION
date_cols = [col for col in df.columns if "date" in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# (C) GEOGRAPHY & OTHER COLUMNS (Replacing null with a string)
df['admin3'] = df['admin3'].fillna("Unknown Township")
df['civilian_targeting'] = df['civilian_targeting'].fillna("Not Targeted")

# (D) TAGS & CROWD CLEANING (Redundancies Removed)
# exract data or cells which contain the word "crowd_size = " in the tag colummn
df['crowd_tag_clean'] = df['tags'].str.extract(r'crowd size=([^;]+)', flags=re.IGNORECASE)[0].fillna("No Crowd Data")

# data or cells which contain the word "crowd_size = " , we extract more (which are numbers)
df['crowd_size_number'] = df['crowd_tag_clean'].str.extract(r'(\d+)').astype(float).fillna(0)

# Apply function only one time
df['crowd_category'] = df['crowd_tag_clean'].apply(truly_dynamic_categorizer)

# (E) ACTOR 2 DYNAMIC CLEANING
EVENT_ACTOR_MAP = {
    'protest': 'Unopposed Protest',
    'battle': 'Unknown Opponent (Missing Data)',
    'strategic development': 'Strategic Movement (No Opponent)',
    'explosion': 'Remote Attack (No Direct Opponent)'
}

def dynamic_actor2_cleaner(row):
    if pd.notna(row['actor2']):
        return row['actor2']
    event = str(row['event_type']).lower()
    for key, label in EVENT_ACTOR_MAP.items():
        if key in event:
            return label
    return "No Opponent Identified"

# Making apply and buliding new column (filling null) to make (visulization) that are born from original columns
df['actor2_viz'] = df.apply(dynamic_actor2_cleaner, axis=1)
df['assoc_actor_1_viz'] = df['assoc_actor_1'].fillna("Sole Actor")
df['assoc_actor_2_viz'] = df['assoc_actor_2'].fillna("Sole Actor")
df['inter2_viz'] = df['inter2'].fillna("No Interaction/Single Actor")

# (F) SOCIAL FEATURES EXTRACTION
#def extract_social_features(df):
    #temp_tags = df['tags'].fillna("").str.lower()
    #df['is_women_targeted'] = temp_tags.str.contains('women targeted', na=False)
    #df['women_political_party'] = temp_tags.str.contains('political party', na=False)
    #df['women_girls'] = temp_tags.str.contains('girls', na=False)
    #df['women_relatives'] = temp_tags.str.contains('relatives', na=False)
    #df['is_armed_presence'] = temp_tags.str.contains('armed presence', na=False)
    #return df
def extract_social_features_dynamic(df):
    # column names and keywords that we want to find are put in the dicitonary. So, later, more new data can be hancled. This is only
    #for the column name called tags
    social_map = {
        'is_women_targeted': 'women targeted',
        'women_political_party': 'political party',
        'women_girls': 'girls',
        'women_relatives': 'relatives',
        'is_armed_presence': 'armed presence'
    }

    # Building loops and making new columns
    for col, keyword in social_map.items():
        df[col] = df['tags'].str.contains(keyword, case=False, na=False)

    return df
df = extract_social_features(df)

# (G) DATA INTEGRITY CHECK (Dynamic Watchdog)
existing_keys = EVENT_ACTOR_MAP.keys()
current_events = df['event_type'].str.lower().unique()
missing_events = [e for e in current_events if not any(key in e for key in existing_keys)]

# --- Checking the results---
print("Cleaning Complete (Redundancies Removed)!")
print(f"Total Rows: {len(df)}")
if missing_events:
    print(f" Check Required for New Event Types: {missing_events}")

# Sample Check
print(df[df['is_women_targeted']][['tags', 'is_women_targeted', 'women_girls']].head())

In [ ]:
df.info()

In [ ]:
# Checking alredy-cleaned columns to make ti sure
check_cols = ['actor2_viz', 'assoc_actor_1_viz', 'inter2_viz', 'admin3', 'crowd_tag_clean', 'crowd_category']
print(df[check_cols].isna().sum())

In [ ]:

test_cases = ["hundreds of thousands", "about 50", "ten thousand", "no report"]
for t in test_cases:
    print(f"Text: {t:25} | Category: {truly_dynamic_categorizer(t)}")

In [ ]:
# checking the number of crowd category
print(df['crowd_category'].value_counts())

In [ ]:
# Comparing the original and new created columns(crowd_tag_clean column and crowd_category columns are born from the tags column)
print(df[['tags', 'crowd_tag_clean', 'crowd_category']].sample(5))

In [ ]:
df['tags'].unique()

In [ ]:
# Making summary for the main facts (until now only for tag column)
summary_stats = pd.DataFrame({
    'Category': ['Women Targeted (Total)', 'Political Party Supporters', 'Girls', 'Relatives of Targeted Groups', 'Armed Presence'],
    'Count': [
        df['is_women_targeted'].sum(),
        df['women_political_party'].sum(),
        df['women_girls'].sum(),
        df['women_relatives'].sum(),
        df['is_armed_presence'].sum()
    ]
})

print(summary_stats)

# showing with bars
summary_stats.plot(kind='bar', x='Category', y='Count', title='Social Impact Analysis Summary')

In [ ]:
# checking associate actors columns
# Original and Cleaned (Vizualiztion column for associate actros are compared)
check_assoc = df[['assoc_actor_1', 'assoc_actor_1_viz', 'assoc_actor_2', 'assoc_actor_2_viz']].sample(10)

print("--- Checking Associated Actors (Original vs Cleaned) ---")
print(check_assoc)

# which org contains the most associate actor
print("\n--- Top 10 Associated Actors (1) ---")
# looking the asso actor that is not nll
print(df['assoc_actor_1_viz'].value_counts().head(10))

In [ ]:
# Counting data which does not contain nothing acording to event type
opponent_check = df.groupby('event_type')['actor2_viz'].apply(lambda x: (x == 'No Opponent Identified').sum()).reset_index()
opponent_check.columns = ['Event Type', 'No Opponent Count']


total_events = df['event_type'].value_counts().reset_index()
total_events.columns = ['Event Type', 'Total Count']

# Percentage of no oppoenent
final_check = pd.merge(opponent_check, total_events, on='Event Type')
final_check['No Opponent %'] = (final_check['No Opponent Count'] / final_check['Total Count'] * 100).round(2)

print("--- Proving Actor 2 Logic by Event Type ---")
print(final_check.sort_values(by='No Opponent %', ascending=False))

In [ ]:
df.info()

In [ ]:
import os
import pandas as pd
from datetime import datetime

def export_final_dataset(df):
    #  Clean Dataset with current date(Format: Year-Month-Day)
    # Example- myanmar_conflict_clean_2026-04-01.csv
    timestamp = datetime.now().strftime("%Y-%m-%d")
    filename = f"myanmar_conflict_clean_{timestamp}.csv"

    try:
        # ၂။ Exporting as csv
        df.to_csv(filename, index=False)
        print("-" * 50)
        print(f"Dynamic Export Successful!")
        print(f"File Name: {filename}")
        print(f"Saved at: {datetime.now().strftime('%p, %d %B %Y')}")
        print(f"Total Records: {len(df):,} rows")
        print("-" * 50)

        # ၃။ It will be automatically downloaded if it is used in google colab
        # (if local IDE, this part will be skipped)
        try:
            from google.colab import files
            files.download(filename)
            print("Download starting in your browser...")
        except ImportError:
            # Local IDE (VS Code, etc.) showing file location for other ide
            full_path = os.path.abspath(filename)
            print(f" Local IDE detected. File saved at:\n{full_path}")

    except Exception as e:
        print(f" Export Error: {e}")

export_final_dataset(df)